# Aprendizagem de Máquina II — Rede RBF
**Aluno: Ariel Ladislau Reises**  
Adaptação da atividade de 03/09/2026.

Execute todas as células no Google Colab ou Jupyter. Bibliotecas: NumPy, pandas, Matplotlib e scikit-learn. O notebook mantém os mesmos dados e a mesma divisão do original.

## Como funciona
Uma rede RBF (Radial Basis Function) possui uma camada de funções gaussianas e uma camada de saída. Cada neurônio oculto responde à proximidade de um centro:

$$\phi_j(x)=\exp(-\gamma \|x-c_j\|^2),\qquad \gamma=\frac{1}{2\sigma^2}.$$

Aqui os centros são aprendidos por K-Means **somente no treino**. A camada de saída usa uma combinação linear das ativações seguida de sigmoide, treinada por descida do gradiente com entropia cruzada e regularização L2. É uma rede RBF explícita; não é uma SVM com kernel RBF. Os hiperparâmetros são fixados antes da avaliação do teste.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

plt.rcParams.update({'figure.figsize': (7, 5), 'font.size': 11})
X, y = make_moons(n_samples=300, noise=0.20, random_state=42)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='plasma', edgecolors='white')
plt.xlabel('Característica 1')
plt.ylabel('Característica 2')
plt.title('Problema de classificação não linear — duas luas')
plt.show()

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_treino_scaled = scaler.fit_transform(X_treino)
X_teste_scaled = scaler.transform(X_teste)
print(f'Treino: {len(y_treino)} amostras | Teste: {len(y_teste)} amostras')

## Modelo linear de referência

In [ ]:
modelo_linear = LogisticRegression(random_state=42)
modelo_linear.fit(X_treino_scaled, y_treino)
previsao_linear = modelo_linear.predict(X_teste_scaled)
acuracia_linear = accuracy_score(y_teste, previsao_linear)
print(f'Acurácia do modelo linear: {acuracia_linear:.2%}')

## Implementação da rede RBF
Os centros e a largura das gaussianas ficam fixos durante o treinamento da saída. Assim, a curva abaixo descreve o aprendizado dos pesos e do bias da saída, não o ajuste dos centros. O gradiente e a perda são calculados sobre todas as amostras de treino a cada iteração.

In [ ]:
class RedeRBF:
    def __init__(self, n_centros=20, gamma=1.0, learning_rate=0.1,
                 n_iter=3000, l2=0.001, random_state=42):
        self.n_centros = n_centros
        self.gamma = gamma
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.l2 = l2
        self.random_state = random_state

    def transform(self, X):
        dist2 = np.sum((np.asarray(X)[:, None, :] - self.centros_[None, :, :]) ** 2, axis=2)
        return np.exp(-self.gamma * dist2)

    @staticmethod
    def _sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        self.kmeans_ = KMeans(n_clusters=self.n_centros, n_init=10,
                             random_state=self.random_state).fit(X)
        self.centros_ = self.kmeans_.cluster_centers_
        phi = self.transform(X)
        self.pesos_ = np.zeros(self.n_centros)
        self.bias_ = 0.0
        self.loss_curve_ = []
        for _ in range(self.n_iter):
            p = self._sigmoid(phi @ self.pesos_ + self.bias_)
            erro = p - y
            self.pesos_ -= self.learning_rate * (phi.T @ erro / len(y) + self.l2 * self.pesos_)
            self.bias_ -= self.learning_rate * np.mean(erro)
            z = phi @ self.pesos_ + self.bias_
            loss = np.mean(np.logaddexp(0, z) - y * z)
            loss += 0.5 * self.l2 * np.sum(self.pesos_ ** 2)
            self.loss_curve_.append(float(loss))
        self.n_iter_ = self.n_iter
        return self

    def predict_proba(self, X):
        p = self._sigmoid(self.transform(X) @ self.pesos_ + self.bias_)
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

modelo_rbf = RedeRBF()
modelo_rbf.fit(X_treino_scaled, y_treino)
previsao_rbf = modelo_rbf.predict(X_teste_scaled)
acuracia_rbf = accuracy_score(y_teste, previsao_rbf)
print(f'Acurácia da rede RBF: {acuracia_rbf:.2%}')
print(classification_report(y_teste, previsao_rbf,
                            target_names=['Classe 0', 'Classe 1'], zero_division=0))

## Avaliação no teste e comparação

In [ ]:
comparacao = pd.DataFrame({
    'Modelo': ['Regressão logística (linear)', 'Rede RBF'],
    'Acurácia treino (%)': [100 * accuracy_score(y_treino, modelo_linear.predict(X_treino_scaled)),
                           100 * accuracy_score(y_treino, modelo_rbf.predict(X_treino_scaled))],
    'Acurácia teste (%)': [100 * acuracia_linear, 100 * acuracia_rbf]})
display(comparacao.round(2))
ConfusionMatrixDisplay.from_predictions(y_teste, previsao_rbf,
    display_labels=['Classe 0', 'Classe 1'], cmap='Blues', colorbar=False)
plt.title('Rede RBF — matriz de confusão no teste')
plt.show()

## Centros, pesos e ativações da camada oculta

In [ ]:
print(f'Iterações da saída: {modelo_rbf.n_iter_}')
print(f'Centros: {modelo_rbf.centros_.shape} (neurônios × características)')
print(f'Pesos da saída: {modelo_rbf.pesos_.shape}')
print(f'Bias da saída: {modelo_rbf.bias_:.4f}')
print(f'Gamma: {modelo_rbf.gamma} | Sigma: {np.sqrt(1 / (2 * modelo_rbf.gamma)):.4f}')
centros_originais = scaler.inverse_transform(modelo_rbf.centros_)
plt.scatter(X_treino[:, 0], X_treino[:, 1], c=y_treino, cmap='plasma', alpha=0.65)
plt.scatter(centros_originais[:, 0], centros_originais[:, 1], marker='X',
            s=130, c='limegreen', edgecolors='black', label='Centros RBF')
plt.xlabel('Característica 1'); plt.ylabel('Característica 2')
plt.title('Centros aprendidos apenas com os dados de treino')
plt.legend(); plt.show()
plt.imshow(modelo_rbf.transform(X_teste_scaled[:20]), aspect='auto', cmap='viridis')
plt.colorbar(label='Ativação gaussiana')
plt.xlabel('Neurônio RBF'); plt.ylabel('Amostra de teste')
plt.title('Ativações para as primeiras 20 amostras de teste')
plt.show()

## Evolução do erro de treinamento

In [ ]:
historico_erros = np.array(modelo_rbf.loss_curve_)
melhor_epoca = int(np.argmin(historico_erros))
plt.plot(np.arange(1, len(historico_erros) + 1), historico_erros, color='#0ea5e9')
plt.scatter(melhor_epoca + 1, historico_erros[melhor_epoca], c='green',
            label=f'Menor perda de treino: iteração {melhor_epoca + 1}')
plt.xlabel('Iteração'); plt.ylabel('Entropia cruzada + penalidade L2')
plt.title('Rede RBF — aprendizado da camada de saída')
plt.grid(alpha=0.3); plt.legend(); plt.show()
print(f'Perda inicial: {historico_erros[0]:.4f} | Perda final: {historico_erros[-1]:.4f}')

## Fronteiras de decisão
Os modelos recebem pontos normalizados pelo mesmo scaler ajustado no treino. As coordenadas do gráfico permanecem na escala original; os pontos exibidos são do conjunto de teste.

In [ ]:
x1, x2 = np.meshgrid(
    np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 220),
    np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 220))
grade_scaled = scaler.transform(np.column_stack([x1.ravel(), x2.ravel()]))
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
for ax, modelo, titulo in zip(axes, [modelo_linear, modelo_rbf],
                              ['Modelo linear', 'Rede RBF']):
    prob = modelo.predict_proba(grade_scaled)[:, 1].reshape(x1.shape)
    ax.contourf(x1, x2, prob, levels=np.linspace(0, 1, 21), cmap='plasma', alpha=0.5)
    ax.contour(x1, x2, prob, levels=[0.5], colors='black', linewidths=1.5)
    ax.scatter(X_teste[:, 0], X_teste[:, 1], c=y_teste, cmap='plasma', edgecolors='black')
    ax.set_title(titulo); ax.set_xlabel('Característica 1')
axes[0].set_ylabel('Característica 2')
plt.tight_layout(); plt.show()

## Interpretação dos resultados
- A regressão logística produz uma fronteira linear nas características normalizadas.
- A rede RBF combina respostas locais das gaussianas para construir uma fronteira curva.
- `n_centros` controla a quantidade de neurônios ocultos. `gamma` maior produz respostas mais estreitas; menor produz respostas mais amplas.
- A comparação usa exatamente o mesmo conjunto de teste. Um resultado melhor nesta divisão não comprova superioridade em todos os problemas.
- A menor perda de treino não determina a melhor generalização. Para ajustar centros, gamma ou regularização, use validação dentro do treino e deixe o teste reservado à avaliação final.
- Na RBF, a matriz de centros e os pesos da saída substituem as matrizes de pesos entre camadas da MLP original.